# Rprop 详解：独立步长 · 符号驱动 · 仅限确定性梯度

## 0. 基本档案

| 项目 | 内容 |
|------|------|
| **全称** | **Resilient Backpropagation**（弹性反向传播） |
| **中文译名** | 弹性反向传播 / 弹性传播 |
| **提出者** | Martin Riedmiller 和 Heinrich Braun |
| **提出年份** | 1993 年 |
| **发表会议** | IEEE International Conference on Neural Networks (ICNN 1993) |
| **学术简称** | Rprop |
| **深度学习社区常用名** | Rprop（无别名） |
| **所属家族** | 自适应学习率方法（每个参数独立步长） |
| **核心创新** | 用梯度符号决定方向，用梯度符号的一致性调节步长大小 |
| **理论收敛性** | 能保证收敛到梯度为零的临界点（可能是局部极小点或鞍点）；但对跨越浅谷（快速通过平坦区域）和逃离鞍点**没有任何理论保证** |
| **致命局限** | 仅适用于确定性梯度（全批量），对 Mini-Batch 噪声极度敏感 |

> 对应原文附录术语对照表：**Rprop | Rprop | Braun & Riedmiller, 1993**

## 1. 核心洞察：梯度幅值比 ≠ 最优步长比

### 1.1 梯度的"最优性"——仅限于"无穷小步长"

在某个参数点 $\theta$，负梯度方向 $-\nabla f(\theta)$ 是**函数值下降最快的方向**。这是微积分的基本定理：

$$
\min_{\|d\|=1} \frac{\partial f}{\partial d} = -\nabla f
$$

这个结论的成立条件是：**步长趋近于无穷小（$\alpha \to 0$）**。在步长无穷小的极限下，负梯度方向确实是唯一最优的。

### 1.2 梯度幅值比是什么？——局部敏感度之比

梯度向量的各分量是函数对每个参数的偏导数：

$$
g_{t,i} = \frac{\partial f}{\partial \theta_i}
$$

这个值告诉我们的是：**在当前点，如果只动这一个参数，函数值变化有多快**。梯度幅值比（如 $1:100$）就是**各参数的敏感度之比**。

### 1.3 敏感度之比 ≠ 最优步长配比

考虑一个二次函数：

$$
f(\theta_1, \theta_2) = 0.1\theta_1^2 + 10\theta_2^2
$$

在点 $\theta = (1.0, 0.1)$ 处，梯度为：

$$
g = (0.2, 2.0)
$$

梯度幅值比是 **$10:1$**（$\theta_2$ 是 $\theta_1$ 的 $10$ 倍）。

**如果梯度幅值比就是"最优步长配比"**，那么最优更新应该是：

$$
\Delta\theta = (-0.2\alpha, -2.0\alpha)
$$

即 $\theta_2$ 走 $10$ 倍于 $\theta_1$ 的距离。

**但实际上，从该点到最优点的精确最短路径是直线走到原点：**

$$
\Delta\theta = (-1.0, -0.1)
$$

即 $\theta_1$ 和 $\theta_2$ **各走 $1$ 和 $0.1$**，步长配比是 **$10:1$**。

| 方案 | $\theta_1$ 移动 | $\theta_2$ 移动 | 能否到达原点？ |
|------|---------|---------|--------------|
| 梯度幅值比 ($10:1$) | $-0.2\alpha$ | $-2.0\alpha$ | 需要 $\alpha=5$ 和 $\alpha=0.05$，**矛盾** |
| 精确最优 ($10:1$) | $-1.0$ | $-0.1$ | **一步到达** |

**结论**：梯度幅值比（$10:1$）是"敏感度之比"，而最优步长配比（$10:1$）是"协同抵达最优所需的比例"。两者数值虽然恰好相同，但背后的逻辑完全不同——梯度幅值比告诉的是"当前点各方向的敏感度"，而最优步长配比告诉的是"到达最优所需的移动量比例"。在更复杂的地形中，这两者可能南辕北辙——这正是 Rprop 要修正的问题。

### 1.4 核心总结：梯度大小与最优步长成反比

| 梯度大 | 梯度小 |
|--------|--------|
| 函数对该参数**极其敏感** | 函数对该参数**不敏感** |
| 在该方向上**稍微动一点**就会改变损失 | 在该方向上**需要大动**才能改变损失 |
| ✅ 策略：步长应该**小**（避免跨过最优） | ✅ 策略：步长应该**大**（才能有效推进） |
| ❌ GD 的做法：步长**大** | ❌ GD 的做法：步长**小** |

**结论**：梯度的大小与"最优步长"是**反相关**关系，而非正相关！

### 1.5 为什么 GD 的"梯度比例"在有限步长下失效？

如果步长是无穷小（$\eta \to 0$），负梯度方向确实是下降最快的方向，GD 的梯度比例是正确的。

**但在实际优化中，我们走的是有限步长（大步）**。走了一步之后，地形已经变了，原来的方向不再是最优的。在狭长山谷中，如果保持梯度比例（$\theta_2$ 走 $\theta_1$ 的 $10$ 倍），$\theta_2$ 会瞬间跨过谷底，撞到对面的山壁，产生锯齿状震荡。

**优化的终极目标不是"每一步都走下降最快的方向"，而是"用最少的步数走到谷底"。** 为了这个目标，我们需要**打破梯度比例**，让各参数以不同的节奏协同前进。

### 1.6 Rprop 的核心逻辑：只用符号，不用大小

基于上述洞察，Rprop 的设计直指问题核心：

> **Rprop 彻底抛弃了梯度幅值，仅保留梯度符号。**

对比两种算法的参数更新公式：

| 算法 | 更新公式 | 使用的信息 |
|------|---------|-----------|
| **梯度下降 (GD)** | $\Delta\theta = -\eta \times g$ | 梯度符号 **+** 梯度大小 |
| **Rprop** | $\Delta\theta = -\text{sign}(g) \times \Delta$ | **只**用梯度符号，不用大小 |

Rprop 的设计逻辑是：

- **梯度大**（敏感）→ 需要**小步长** → 通过符号反转频繁将 $\Delta$ 收缩 $0.5$ 倍
- **梯度小**（不敏感）→ 需要**大步长** → 通过符号持续一致将 $\Delta$ 放大 $1.2$ 倍

用一句话概括：

> **梯度幅值 → 只告诉方向（用符号）**  
> **梯度符号的一致性 → 决定步长**

这正是 Rprop 的"符号驱动"逻辑背后的数学直觉。

## 2. Rprop 在演进链条中的位置

```
基础 GD (统一 lr) 
   │
   ├── 问题：统一学习率无法适配不同参数尺度
   │      ↓
   │   Rprop  (Riedmiller & Braun, 1993)
   │      ├── 每个参数独立维护步长 Δᵢ
   │      ├── 只使用梯度符号 sign(gᵢ) 决定方向
   │      └── 局限：依赖确定性梯度，噪声下符号频繁抖动
   │           ↓
   │        Adagrad  (Duchi et al., 2011)
   │           ├── 用梯度平方累加替代符号，容忍噪声
   │           └── 但引入新问题：训练后期强制停滞
   │                ↓
   │             RMSprop / Adam (用EMA替代累加和)
```

**核心定位**：Rprop 是**第一个真正意义上为每个参数独立控制更新步长**的算法，它彻底抛弃了"全局学习率"的枷锁，但代价是 **不能用于 Mini-Batch 随机梯度**。

**命名由来**："Resilient" 意为"有弹性的、能快速恢复的"，指该算法在面对不同参数时能弹性地调整各自的步长，不受梯度幅值大小的影响。

## 3. 迭代规则（完整决策逻辑）

对每个参数 $i$，维护一个独立的步长 $\Delta_i$（初始值通常设为 $0.01$ 或 $0.1$）。

**典型超参数**：

| 参数 | 典型值 | 含义 |
|------|--------|------|
| $\eta^+$ | $1.2$ | 符号一致时步长放大倍数（$>1$），踩油门 |
| $\eta^-$ | $0.5$ | 符号反转时步长缩小倍数（$<1$），踩刹车 |
| $\Delta_{\min}$ | $10^{-6}$ | 步长下限（防止收缩为 $0$） |
| $\Delta_{\max}$ | $50$ | 步长上限（防止爆炸） |

### 3.1 步长更新（根据梯度符号一致性）

**符号一致 → 踩油门（$\eta^+ > 1$）**

$$
\text{if } g_{t,i} \times g_{t-1,i} > 0 \quad \Rightarrow \quad \Delta_i = \min(\Delta_i \times \eta^+, \; \Delta_{\max})
$$

| 问题 | 答案 |
|------|------|
| 条件是什么？ | 上一次和这一次梯度方向相同（都是负，或都是正） |
| 说明什么？ | 你一直沿着同一个方向下坡，**还没跨过谷底** |
| Rprop 怎么做？ | 步子再大一点，$\Delta = \Delta \times \eta^+$（$\eta^+>1$）加速前进 |
| 为什么有 $\min$？ | 防止步长无限膨胀，设个天花板 $\Delta_{\max}$，比如 $50$ |

---

**符号反转 → 踩刹车（$\eta^- < 1$）**

$$
\text{if } g_{t,i} \times g_{t-1,i} < 0 \quad \Rightarrow \quad \Delta_i = \max(\Delta_i \times \eta^-, \; \Delta_{\min})
$$

| 问题 | 答案 |
|------|------|
| 条件是什么？ | 上一次是负梯度，这一次变成了正梯度（或反过来） |
| 说明什么？ | **跨过了谷底！** 从下坡变成了上坡 |
| Rprop 怎么做？ | 步子太大了，$\Delta = \Delta \times \eta^-$（$\eta^-<1$）刹车 |
| 为什么有 $\max$？ | 防止步长缩到 $0$，设个地板 $\Delta_{\min}$，比如 $10^{-6}$ |

---

**梯度为零（极少发生）**

$$
\text{if } g_{t,i} \times g_{t-1,i} = 0 \quad \Rightarrow \quad \Delta_i \text{ 保持不变}
$$

### 3.2 参数更新（仅用符号，Rprop 专用）

$$
\Delta\theta_{t,i} = -\text{sign}(g_{t,i}) \times \Delta_i
$$

**注意**：这是 Rprop 的参数更新公式，使用**梯度的符号**乘以**独立步长 $\Delta$**，而非梯度本身。

> 对比标准梯度下降：$\Delta\theta = -\alpha \times g$（使用梯度大小 × 全局学习率）

### 3.3 最关键的理解：Rprop 每步做两件事

| 顺序 | 操作 | 作用 |
|------|------|------|
| **第 1 步** | 根据符号一致性更新 $\Delta$ | 决定"**下次**走多远" |
| **第 2 步** | $\Delta\theta = -\text{sign}(g) \times \Delta$ | 用**当前的** $\Delta$ 更新参数 |

### 3.4 用具体数字走一遍

假设某个参数 $\theta$，初始 $\Delta = 0.1$。

**Rprop 参数更新公式**：$\Delta\theta = -\text{sign}(g) \times \Delta$

**注意**：这是 Rprop 的专用公式，与标准梯度下降 $\Delta\theta = -\alpha \times g$ 完全不同。

**第 0 步计算过程**：
- $\text{sign}(-0.2) = -1$
- $\Delta\theta = -(-1) \times 0.1 = +0.1$

| 迭代 | 位置 | 梯度 $g$ | $g \times g_{\text{prev}}$ | $\Delta$ 更新 | 新 $\Delta$ | $\Delta\theta = -\text{sign}(g) \times \Delta$ | 新位置 |
|------|------|--------|----------|--------|------|---------------------------------------------|--------|
| 0 | $1.0$ | $-0.2$ | 无历史 | 保持 | $0.1$ | $+0.1$ | **$0.9$** |
| 1 | $0.9$ | $-0.18$ | $>0$（一致） | $\times 1.2$ | **$0.12$** | $+0.12$ | **$0.78$** |
| 2 | $0.78$ | $-0.156$ | $>0$（一致） | $\times 1.2$ | **$0.144$** | $+0.144$ | **$0.636$** |
| 3 | $0.636$ | $-0.127$ | $>0$（一致） | $\times 1.2$ | **$0.173$** | $+0.173$ | **$0.463$** |
| 4 | $0.463$ | $-0.093$ | $>0$（一致） | $\times 1.2$ | **$0.207$** | $+0.207$ | **$0.256$** |

**关键观察**：只要 $g_t \times g_{t-1} > 0$，$\Delta$ 就不断 $\times 1.2$，步长**指数级增长**，不是"慢慢试"，而是在"猛冲"！

**Rprop vs 标准 GD 参数更新对比**：

| 对比项 | 标准梯度下降 (GD) | Rprop |
|--------|------------------|-------|
| 参数更新公式 | $\Delta\theta = -\alpha \times g$ | $\Delta\theta = -\text{sign}(g) \times \Delta$ |
| 第 0 步更新量 | $-0.01 \times (-0.2) = +0.002$ | $-(-1) \times 0.1 = +0.1$ |
| 使用梯度**大小**？ | ✅ 是 | ❌ 否 |
| 使用梯度**符号**？ | ❌ 否 | ✅ 是 |

## 4. 原始 Rprop vs iRprop（改进版）：为什么需要修正？

### 4.1 原始 Rprop 的缺陷：只刹车，不回退

当 $g_t \times g_{t-1} < 0$ 触发时，原始 Rprop 只做一件事：

| 步骤 | 操作 | 目的 |
|------|------|------|
| **1** | $\Delta = \Delta \times \eta^-$（$\eta^-=0.5$） | **刹车**：步长砍半 |

然后**继续正常更新参数**：$\theta = \theta - \text{sign}(g) \times \Delta$

**问题**：参数已经跨过谷底了，但没有回退。用砍半后的步长继续往前走，虽然步长小了，但**方向还是错的**——参数仍然在谷底的"另一侧"，下次更新时梯度方向再次反转，又会触发刹车。

**结果**：在陡峭方向上反复触发刹车，步长被连续砍半，但参数从未真正稳定在谷底，而是在谷底两侧**来回震荡**。

### 4.2 iRprop 的修正：三步制动

当 $g_t \times g_{t-1} < 0$ 触发时，iRprop 执行三步：

| 步骤 | 操作 | 目的 |
|------|------|------|
| **1** | $\Delta = \Delta \times \eta^-$（$\eta^-=0.5$） | **急刹**：下次走小一点 |
| **2** | $\theta = \theta_{\text{old}}$ | **回退**：撤销跨过谷底的那一步 |
| **3** | $g_t = 0$ | **防连刹**：避免下一次又触发 $\times 0.5$ |

> **形象比喻**：你在黑暗中下台阶，一脚踩空跨过了一级台阶。
> - **原始 Rprop**：不管，继续往前走（又跨过，又踩空）→ 在台阶边缘反复踩空（震荡）
> - **iRprop**：收脚回来（回退），缩小步幅（步长砍半），再稳稳踩下去 → 平稳落地（收敛）

### 4.3 为什么 iRprop 要"回退到上一步"？

关键在于：**当 $g_t \times g_{t-1} < 0$ 触发时，当前梯度 $g$ 的方向已经反转了。**

| 情况 | $g_{t-1}$ | $g_t$ | $\text{sign}(g_t)$ | $\theta - \text{sign}(g_t) \times \Delta$ | 效果 |
|------|---------|-----|-----------|------------------------------------------|------|
| 下坡中 | 负 | 负 | $-1$ | $\theta + \Delta$ | 继续下坡（正常前进） |
| **跨过谷底** | 负 | **正** | **$+1$** | **$\theta - \Delta$** | **往回走**（回退） |

所以 iRprop 的 `$\theta = \theta - \text{sign}(g) \times \Delta$` 当 $g$ 为正时，是**往回退**，不是继续前进。这正是"回退"的数学表达。

### 4.4 三步制动的完整逻辑链

```
发现跨过谷底（g_t × g_{t-1} < 0）
    │
    ▼
第一步：急刹（Δ = Δ × 0.5）        ← 下次走小一点
    │
    ▼
第二步：回退（theta - sign(g) × Δ） ← 用缩小的步长往回走
    │
    ▼
第三步：置零（g = 0）               ← 防止连续急刹
```

| 只做第一步 | 只做前两步 | 三步都做（iRprop） |
|-----------|-----------|-------------------|
| 步长小了，但参数还在谷底外侧 | 参数回到谷底附近，步长也小了 | ✅ 参数稳定 + 步长稳定 + 不再连续刹车 |
| ⚠️ 下一次还可能跨过（因为位置不对） | ⚠️ 下次又触发刹车（连续刹车） | ✅ 完美 |

### 4.5 原始 Rprop vs iRprop 完整对比表

| 版本 | 符号反转时的处理 | 效果 |
|------|-----------------|------|
| **原始 Rprop (1993)** | $\Delta = \Delta \times \eta^-$，**不做回退**，**不做梯度置零** | 可能连续刹车导致步长缩死，参数震荡 |
| **iRprop（改进版）** | $\Delta = \Delta \times \eta^-$，**回退参数**，**梯度置零** | 更稳定，真正收敛到谷底 |

**本教程代码中的实现是 iRprop 版本**，因为它更稳定，也是大多数现代深度学习框架采用的版本。

## 5. 算例实验：Rprop 在狭长山谷上的表现

### 5.1 实验设置

目标函数（狭长椭圆）：

$$
f(\theta_1, \theta_2) = 0.1\theta_1^2 + 10\theta_2^2
$$

初始点 $(\theta_1, \theta_2) = (1.0, 0.1)$，最优值为 $(0,0)$。

该函数的 Hessian 特征值为 $0.2$ 和 $20$，条件数高达 $100$，是典型的"狭长山谷"地形。

- $\theta_1$ 方向的梯度很小（$g_1 = 0.2\theta_1$），需要大步长
- $\theta_2$ 方向的梯度很大（$g_2 = 20\theta_2$），需要小步长

**实验目标**：
1. 对比 Rprop（iRprop）与 GD 在狭长山谷上的优化路径与收敛速度
2. 观察 Rprop 独立步长的演化过程
3. 对比原始 Rprop 与 iRprop 的收敛行为差异

### 5.2 完整代码（包含全部 5 张图及迭代详细过程，一次运行完成）

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 第一部分：全局参数配置（集中设置）
# ============================================================

# ---------- 目标函数参数 ----------
FUNCTION_PARAMS = {
    'a': 0.1,          # θ₁ 方向的系数（平坦方向）
    'b': 10.0,         # θ₂ 方向的系数（陡峭方向）
}

# ---------- Rprop 算法参数 ----------
RPROP_PARAMS = {
    'eta_plus': 1.2,           # 步长增长因子（梯度符号不变时）
    'eta_minus': 0.5,          # 步长衰减因子（梯度符号变化时）
    'delta_max': 50.0,         # 最大步长限制
    'delta_min': 1e-6,         # 最小步长限制
    'delta_init': 0.1,         # 初始步长
    'max_iter': 50,            # 最大迭代次数
}

# ---------- 梯度下降（GD）参数 ----------
GD_PARAMS = {
    'lr': 0.01,                # 学习率
    'max_iter': 50,            # 最大迭代次数
}

# ---------- 初始点参数 ----------
INIT_POINT = {
    'theta0': np.array([1.0, 0.1]),   # 二维问题的初始点
}

# ---------- 绘图参数 ----------
PLOT_PARAMS = {
    'contour_range': (-1.5, 1.5),      # 等高线绘制范围
    'contour_points': 200,             # 等高线网格密度
    'width': 850,                      # 图表宽度
    'height_contour': 650,             # 等高线图高度
    'height_standard': 450,            # 标准图表高度
}


# ============================================================
# 第二部分：目标函数定义（使用集中参数）
# ============================================================

def f(theta):
    """目标函数：f(θ) = a * θ₁² + b * θ₂²"""
    a = FUNCTION_PARAMS['a']
    b = FUNCTION_PARAMS['b']
    return a * theta[0]**2 + b * theta[1]**2


def grad_f(theta):
    """目标函数梯度：∇f = [2a*θ₁, 2b*θ₂]"""
    a = FUNCTION_PARAMS['a']
    b = FUNCTION_PARAMS['b']
    return np.array([2 * a * theta[0], 2 * b * theta[1]])


# ============================================================
# 第三部分：iRprop 算法实现（带详细打印）
# ============================================================

def rprop_irprop_with_details_and_print(theta0, max_iter=None, verbose=True):
    """
    iRprop（改进版 Rprop）—— 打印每一步的详细计算过程
    
    步长更新规则（对应 3.1 节）：
    1. 符号一致（g_t × g_{t-1} > 0）：Δ = min(Δ × η⁺, Δ_max)，踩油门
    2. 符号反转（g_t × g_{t-1} < 0）：Δ = max(Δ × η⁻, Δ_min)，踩刹车 + 回退
    3. 梯度为零：Δ 保持不变
    """
    if max_iter is None:
        max_iter = RPROP_PARAMS['max_iter']
    
    eta_plus = RPROP_PARAMS['eta_plus']          # 1.2
    eta_minus = RPROP_PARAMS['eta_minus']       # 0.5
    delta_max = RPROP_PARAMS['delta_max']       # 50.0
    delta_min = RPROP_PARAMS['delta_min']       # 1e-6
    delta_init = RPROP_PARAMS['delta_init']     # 0.1
    
    theta = theta0.copy()
    delta = np.array([delta_init, delta_init])
    g_prev = grad_f(theta)
    
    history = [theta.copy()]
    delta_history = [delta.copy()]
    loss_history = [f(theta)]
    
    # 详细记录列表
    details = []
    
    # ---- 第 0 步（初始状态） ----
    if verbose:
        print("\n" + "=" * 110)
        print("【第 0 步】初始状态")
        print("=" * 110)
        print(f"  参数位置: θ₁ = {theta[0]:.6f}, θ₂ = {theta[1]:.6f}")
        print(f"  梯度:     g₁ = {g_prev[0]:.6f}, g₂ = {g_prev[1]:.6f}")
        # 修正：将 np.sign 的 float 结果转为 int
        print(f"  梯度符号: sign(g₁) = {int(np.sign(g_prev[0])):+d}, sign(g₂) = {int(np.sign(g_prev[1])):+d}")
        print(f"  初始步长: Δ₁ = {delta[0]:.6f}, Δ₂ = {delta[1]:.6f}")
        print(f"  损失:     Loss = {f(theta):.8f}")
    
    details.append({
        'iter': 0,
        'theta1': theta[0], 'theta2': theta[1],
        'g1': g_prev[0], 'g2': g_prev[1],
        'sign1': int(np.sign(g_prev[0])), 'sign2': int(np.sign(g_prev[1])),
        'delta1': delta[0], 'delta2': delta[1],
        'event1': '初始', 'event2': '初始',
        'theta1_after': theta[0], 'theta2_after': theta[1],
        'delta1_after': delta[0], 'delta2_after': delta[1],
        'loss': f(theta)
    })

    # ---- 迭代循环 ----
    for t in range(1, max_iter):
        g = grad_f(theta)
        theta_old = theta.copy()
        delta_old = delta.copy()
        event1, event2 = '', ''
        
        if verbose:
            print("\n" + "=" * 110)
            print(f"【第 {t} 步】迭代计算")
            print("=" * 110)
            print(f"  当前位置: θ₁ = {theta[0]:.6f}, θ₂ = {theta[1]:.6f}")
            print(f"  当前梯度: g₁ = {g[0]:.6f}, g₂ = {g[1]:.6f}")
            # 修正：将 np.sign 的 float 结果转为 int
            print(f"  梯度符号: sign(g₁) = {int(np.sign(g[0])):+d}, sign(g₂) = {int(np.sign(g[1])):+d}")
            print(f"  上一步梯度的符号: sign(g₁_prev) = {int(np.sign(g_prev[0])):+d}, sign(g₂_prev) = {int(np.sign(g_prev[1])):+d}")
            print(f"  当前步长: Δ₁ = {delta[0]:.6f}, Δ₂ = {delta[1]:.6f}")
            print()
            print("  ─── 步长更新（根据梯度符号一致性）───")
        
        # 对每个参数独立处理
        for i in range(len(theta)):
            # 判断符号一致性
            sign_product = g[i] * g_prev[i]
            param_name = 'θ₁' if i == 0 else 'θ₂'
            
            if verbose:
                print(f"\n  参数 {param_name}:")
                print(f"    g_t × g_{{t-1}} = {g[i]:.6f} × {g_prev[i]:.6f} = {sign_product:.6f}")
            
            if g[i] * g_prev[i] > 0:
                # 符号一致 → 踩油门
                delta_old_i = delta[i]
                delta[i] = min(delta[i] * eta_plus, delta_max)
                event = f'油门×{eta_plus:.1f}'
                
                if verbose:
                    print(f"    ✅ 符号一致 ({sign_product:.6f} > 0) → 踩油门")
                    print(f"       Δ_new = min(Δ_old × η⁺, Δ_max)")
                    print(f"       = min({delta_old_i:.6f} × {eta_plus:.1f}, {delta_max:.1f})")
                    print(f"       = {delta[i]:.6f}")
                
                if i == 0: event1 = event
                else: event2 = event
                
            elif g[i] * g_prev[i] < 0:
                # 符号反转 → 踩刹车 + 回退
                delta_old_i = delta[i]
                delta[i] = max(delta[i] * eta_minus, delta_min)
                event = f'刹车×{eta_minus:.1f}+回退'
                
                if verbose:
                    print(f"    ❌ 符号反转 ({sign_product:.6f} < 0) → 跨过了谷底！")
                    print(f"       第一步（刹车）：")
                    print(f"       Δ_new = max(Δ_old × η⁻, Δ_min)")
                    print(f"       = max({delta_old_i:.6f} × {eta_minus:.1f}, {delta_min:.1e})")
                    print(f"       = {delta[i]:.6f}")
                
                # iRprop 回退
                theta_before_rollback = theta[i]
                theta[i] = theta[i] - np.sign(g[i]) * delta[i]
                
                if verbose:
                    print(f"       第二步（回退）：")
                    print(f"       θ_new = θ_old - sign(g) × Δ_new")
                    print(f"       = {theta_before_rollback:.6f} - ({int(np.sign(g[i])):+d}) × {delta[i]:.6f}")
                    print(f"       = {theta[i]:.6f}")
                    print(f"       第三步（防连刹）：梯度 g = 0")
                
                if i == 0: event1 = event
                else: event2 = event
                
                # 防连刹：梯度置零
                g[i] = 0
                
            else:
                # 梯度为零（极少发生）
                if verbose:
                    print(f"    ⚪ 梯度为零 → 步长保持不变")
                if i == 0: event1 = '保持'
                else: event2 = '保持'
        
        # ---- 参数更新 ----
        if verbose:
            print()
            print("  ─── 参数更新（用当前步长沿梯度反方向移动）───")
            print(f"  更新前: θ₁ = {theta_old[0]:.6f}, θ₂ = {theta_old[1]:.6f}")
            print()
        
        # 用更新后的步长更新参数
        theta_before_update = theta.copy()
        theta = theta - np.sign(g) * delta
        
        if verbose:
            for i in range(2):
                param_name = 'θ₁' if i == 0 else 'θ₂'
                delta_name = 'Δ₁' if i == 0 else 'Δ₂'
                print(f"  {param_name}:")
                print(f"    θ_new = θ_old - sign(g) × {delta_name}")
                print(f"    = {theta_before_update[i]:.6f} - ({int(np.sign(g[i])):+d}) × {delta[i]:.6f}")
                print(f"    = {theta[i]:.6f}")
            print()
        
        # 记录本步迭代
        details.append({
            'iter': t,
            'theta1': theta_old[0], 'theta2': theta_old[1],
            'g1': g[0] if g[0] != 0 else g_prev[0],
            'g2': g[1] if g[1] != 0 else g_prev[1],
            'sign1': int(np.sign(g[0])) if g[0] != 0 else int(np.sign(g_prev[0])),
            'sign2': int(np.sign(g[1])) if g[1] != 0 else int(np.sign(g_prev[1])),
            'delta1': delta_old[0], 'delta2': delta_old[1],
            'event1': event1, 'event2': event2,
            'theta1_after': theta[0], 'theta2_after': theta[1],
            'delta1_after': delta[0], 'delta2_after': delta[1],
            'loss': f(theta)
        })
        
        if verbose:
            print(f"  ─── 更新结果 ───")
            print(f"  新参数: θ₁ = {theta[0]:.6f}, θ₂ = {theta[1]:.6f}")
            print(f"  新步长: Δ₁ = {delta[0]:.6f}, Δ₂ = {delta[1]:.6f}")
            print(f"  损失:   Loss = {f(theta):.10f}")
        
        history.append(theta.copy())
        delta_history.append(delta.copy())
        loss_history.append(f(theta))
        g_prev = g.copy()
    
    if verbose:
        print("\n" + "=" * 110)
        print("【迭代完成】")
        print("=" * 110)
        print(f"  最终参数: θ₁ = {theta[0]:.10f}, θ₂ = {theta[1]:.10f}")
        print(f"  最终步长: Δ₁ = {delta[0]:.10f}, Δ₂ = {delta[1]:.10f}")
        print(f"  最终损失: Loss = {f(theta):.12f}")
    
    return np.array(history), np.array(delta_history), np.array(loss_history), pd.DataFrame(details)


def rprop_original(theta0, max_iter=None):
    """原始 Rprop（无梯度符号反转回退）"""
    if max_iter is None:
        max_iter = RPROP_PARAMS['max_iter']
    
    eta_plus = RPROP_PARAMS['eta_plus']
    eta_minus = RPROP_PARAMS['eta_minus']
    delta_max = RPROP_PARAMS['delta_max']
    delta_min = RPROP_PARAMS['delta_min']
    delta_init = RPROP_PARAMS['delta_init']
    
    theta = theta0.copy()
    delta = np.array([delta_init, delta_init])
    g_prev = grad_f(theta)
    history = [theta.copy()]
    delta_history = [delta.copy()]
    loss_history = [f(theta)]
    
    for t in range(1, max_iter):
        g = grad_f(theta)
        for i in range(len(theta)):
            if g[i] * g_prev[i] > 0:
                delta[i] = min(delta[i] * eta_plus, delta_max)
            elif g[i] * g_prev[i] < 0:
                delta[i] = max(delta[i] * eta_minus, delta_min)
        theta = theta - np.sign(g) * delta
        history.append(theta.copy())
        delta_history.append(delta.copy())
        loss_history.append(f(theta))
        g_prev = g.copy()
    
    return np.array(history), np.array(delta_history), np.array(loss_history)


def gd(theta0, lr=None, max_iter=None):
    """标准梯度下降"""
    if lr is None:
        lr = GD_PARAMS['lr']
    if max_iter is None:
        max_iter = GD_PARAMS['max_iter']
    
    theta = theta0.copy()
    history = [theta.copy()]
    loss_history = [f(theta)]
    for _ in range(max_iter - 1):
        theta = theta - lr * grad_f(theta)
        history.append(theta.copy())
        loss_history.append(f(theta))
    
    return np.array(history), np.array(loss_history)


# ============================================================
# 第四部分：运行主实验（打印详细计算过程）
# ============================================================

print("=" * 70)
print("Rprop 详解 —— iRprop 详细计算过程演示")
print("=" * 70)

theta0 = INIT_POINT['theta0']

print("\n📐 目标函数: f(θ) = 0.1 × θ₁² + 10 × θ₂²")
print(f"   初始点: θ₀ = ({theta0[0]:.1f}, {theta0[1]:.1f})")
print(f"   初始损失: f(θ₀) = {f(theta0):.4f}")
print(f"   初始梯度: ∇f(θ₀) = ({grad_f(theta0)[0]:.1f}, {grad_f(theta0)[1]:.1f})")
print("\n" + "-" * 70)
print("📋 Rprop 超参数:")
print(f"   η⁺ = {RPROP_PARAMS['eta_plus']:.1f} (符号一致时步长放大倍数)")
print(f"   η⁻ = {RPROP_PARAMS['eta_minus']:.1f} (符号反转时步长缩小倍数)")
print(f"   Δ_init = {RPROP_PARAMS['delta_init']:.1f} (初始步长)")
print(f"   Δ_min = {RPROP_PARAMS['delta_min']:.0e} (步长下限)")
print(f"   Δ_max = {RPROP_PARAMS['delta_max']:.1f} (步长上限)")
print("=" * 70)

# 运行 iRprop 并打印详细过程
rprop_traj, rprop_delta, rprop_loss, details_df = rprop_irprop_with_details_and_print(
    theta0, verbose=True
)

# 运行 GD 用于对比
gd_traj, gd_loss = gd(theta0)

print("\n" + "=" * 70)
print("【优化结果对比】")
print("-" * 50)
print(f"iRprop 终点: θ₁={rprop_traj[-1][0]:.6f}, θ₂={rprop_traj[-1][1]:.6f}, Loss={f(rprop_traj[-1]):.8f}")
print(f"GD 终点:     θ₁={gd_traj[-1][0]:.6f}, θ₂={gd_traj[-1][1]:.6f}, Loss={f(gd_traj[-1]):.8f}")


# ============================================================
# 第五部分：输出迭代详细表格
# ============================================================

print("\n" + "=" * 70)
print("迭代详细过程表格")
print("=" * 70)

# 表1：平坦方向 θ₁
df1 = details_df[['iter', 'theta1', 'g1', 'sign1', 'event1', 'delta1', 'delta1_after', 'theta1_after', 'loss']].copy()
df1.columns = ['Iter', 'θ₁_before', 'g₁', 'sign(g₁)', 'Event', 'Δ₁_before', 'Δ₁_after', 'θ₁_after', 'Loss']

print("\n" + "=" * 130)
print("表1：平坦方向 θ₁ 的完整迭代过程 (0~20步)")
print("=" * 130)
print(df1.head(21).to_string(index=False))
print("\n... 后续30步详见完整数据 ...")

# 表2：陡峭方向 θ₂
df2 = details_df[['iter', 'theta2', 'g2', 'sign2', 'event2', 'delta2', 'delta2_after', 'theta2_after', 'loss']].copy()
df2.columns = ['Iter', 'θ₂_before', 'g₂', 'sign(g₂)', 'Event', 'Δ₂_before', 'Δ₂_after', 'θ₂_after', 'Loss']

print("\n" + "=" * 130)
print("表2：陡峭方向 θ₂ 的完整迭代过程 (0~20步)")
print("=" * 130)
print(df2.head(21).to_string(index=False))
print("\n... 后续30步详见完整数据 ...")

# 表3：关键节点对比
print("\n" + "=" * 130)
print("表3：双方向同步对比 —— 步长与符号反转频率 (关键节点)")
print("=" * 130)

key_iters = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 30, 40, 49]
df_compare = details_df[details_df['iter'].isin(key_iters)][['iter', 'theta1', 'theta2', 'delta1_after', 'delta2_after', 'event1', 'event2', 'loss']].copy()
df_compare.columns = ['Iter', 'θ₁', 'θ₂', 'Δ₁', 'Δ₂', 'θ₁ Event', 'θ₂ Event', 'Loss']
print(df_compare.to_string(index=False))

# 统计信息
brake_1_total = details_df['event1'].str.contains('刹车').sum()
brake_2_total = details_df['event2'].str.contains('刹车').sum()
final_d1 = details_df['delta1_after'].iloc[-1]
final_d2 = details_df['delta2_after'].iloc[-1]
peak_d1 = details_df['delta1_after'].max()
peak_d2 = details_df['delta2_after'].max()

print("\n" + "=" * 130)
print("📊 50步统计:")
print(f"  θ₁ 刹车 {brake_1_total} 次 | 步长峰值 {peak_d1:.6f} → {final_d1:.6f} (收缩 {peak_d1/final_d1:.0f} 倍)")
print(f"  θ₂ 刹车 {brake_2_total} 次 | 步长峰值 {peak_d2:.6f} → {final_d2:.6f} (收缩 {peak_d2/final_d2:.0f} 倍)")
print(f"  Δ₁/Δ₂ = {final_d1/final_d2:.2f}:1")
print(f"  最终损失 = {details_df['loss'].iloc[-1]:.10f}")


# ============================================================
# 第六部分：生成 5 张 Plotly 图
# ============================================================

iterations = list(range(len(rprop_loss)))
iters_delta = list(range(len(rprop_delta)))

contour_range = PLOT_PARAMS['contour_range']
contour_points = PLOT_PARAMS['contour_points']
width = PLOT_PARAMS['width']
h_contour = PLOT_PARAMS['height_contour']
h_std = PLOT_PARAMS['height_standard']

x_contour = np.linspace(contour_range[0], contour_range[1], contour_points)
y_contour = np.linspace(contour_range[0], contour_range[1], contour_points)
X, Y = np.meshgrid(x_contour, y_contour)
Z = FUNCTION_PARAMS['a'] * X**2 + FUNCTION_PARAMS['b'] * Y**2

LEGEND_STYLE = dict(x=0.5, y=1.02, xanchor='center', yanchor='bottom', orientation='h', font=dict(size=12), bgcolor='rgba(255,255,255,0.85)')
MARGIN = dict(l=80, r=40, t=80, b=60)
LINE_RED = dict(color='red', width=1.5, dash='solid')
MARKER_RED = dict(size=4, color='red', symbol='circle')
LINE_BLUE = dict(color='blue', width=1.5, dash='dot')
MARKER_BLUE = dict(size=4, color='blue', symbol='square')
AXIS_STYLE = dict(gridcolor='lightgray', gridwidth=0.5, griddash='solid', zeroline=True, zerolinecolor='gray', zerolinewidth=1, showline=True, linecolor='black', linewidth=1)

def make_log_axis(base_range, step=1):
    start, end = base_range
    superscript_map = {'-': '⁻', '0': '⁰', '1': '¹', '2': '²', '3': '³', '4': '⁴', '5': '⁵', '6': '⁶', '7': '⁷', '8': '⁸', '9': '⁹'}
    def to_superscript(num):
        s = str(num)
        return ''.join(superscript_map.get(c, c) for c in s)
    tickvals = [10**i for i in range(start, end + 1, step)]
    ticktext = []
    for i in range(start, end + 1, step):
        if i < 0: ticktext.append(f'10{to_superscript(i)}')
        elif i == 0: ticktext.append('10⁰')
        else: ticktext.append(f'10{to_superscript(i)}')
    return dict(type='log', tickmode='array', tickvals=tickvals, ticktext=ticktext, gridcolor='lightgray', gridwidth=0.5, griddash='solid', zeroline=True, zerolinecolor='gray', zerolinewidth=1, showline=True, linecolor='black', linewidth=1, range=base_range)

LOG_YAXIS_LOSS = make_log_axis([-17, 2], 2)
LOG_YAXIS_DELTA = make_log_axis([-8, 2], 1)

# 图1：迭代路径对比
fig1 = go.Figure()
fig1.add_trace(go.Contour(x=x_contour, y=y_contour, z=Z, colorscale='Viridis', ncontours=25, opacity=0.75, contours=dict(coloring='heatmap'), colorbar=dict(title='Loss'), name='等高线'))
fig1.add_trace(go.Scatter(x=rprop_traj[:, 0], y=rprop_traj[:, 1], mode='lines+markers', name='iRprop', line=LINE_RED, marker=MARKER_RED))
fig1.add_trace(go.Scatter(x=gd_traj[:, 0], y=gd_traj[:, 1], mode='lines+markers', name='GD (lr=0.01)', line=LINE_BLUE, marker=MARKER_BLUE))
fig1.add_trace(go.Scatter(x=[INIT_POINT['theta0'][0]], y=[INIT_POINT['theta0'][1]], mode='markers', name='起点', marker=dict(size=10, color='red', symbol='x')))
fig1.add_trace(go.Scatter(x=[0], y=[0], mode='markers', name='最优点', marker=dict(size=8, color='white', symbol='star', line=dict(color='black', width=1.5))))
fig1.update_layout(title=dict(text='图1：迭代路径对比 —— iRprop vs 固定学习率 GD', font=dict(size=18)), xaxis_title='θ₁', yaxis_title='θ₂', xaxis=AXIS_STYLE, yaxis=AXIS_STYLE, width=width, height=h_contour, hovermode='closest', legend=LEGEND_STYLE, margin=MARGIN)
fig1.show()

**图1 解读：迭代路径对比（iRprop vs GD）**

本图展示了 iRprop 和固定学习率 GD 在狭长山谷地形上的优化轨迹：

- **GD（蓝色方点）**：在 $\theta_2$ 方向（陡峭方向）来回剧烈震荡，呈现明显锯齿状；在 $\theta_1$ 方向（平坦方向）推进缓慢，$50$ 步仍未收敛。
- **iRprop（红色圆点）**：在 $\theta_1$ 方向大步长快速推进，在 $\theta_2$ 方向小步长平稳前进，路径近乎直线地从起点直达最优点。

**关键观察**：iRprop 通过独立步长机制，在平坦方向持续放大步长，在陡峭方向持续压缩步长，从而避免了 GD 的锯齿震荡问题。

In [ ]:
# 图2：损失收敛曲线
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=iterations, y=rprop_loss, mode='lines+markers', name='iRprop', line=LINE_RED, marker=MARKER_RED))
fig2.add_trace(go.Scatter(x=iterations, y=gd_loss, mode='lines+markers', name='GD (lr=0.01)', line=LINE_BLUE, marker=MARKER_BLUE))

# 方法1：添加不可见的散点图作为图例项（推荐）
fig2.add_trace(go.Scatter(
    x=[None], y=[None],  # 不可见的数据点
    mode='lines',
    name='最优 y=0',
    line=dict(color='black', dash='dot', width=2),
    showlegend=True
))

# 添加参考线（不显示图例）
fig2.add_hline(y=0, line_dash="dot", line_color="black", line_width=2)

fig2.update_layout(
    title=dict(text='图2：损失函数收敛曲线对比（iRprop vs GD）', font=dict(size=18)),
    xaxis_title='迭代次数',
    yaxis_title='Loss',
    xaxis=AXIS_STYLE,
    width=width,
    height=h_std,
    hovermode='x',
    legend=LEGEND_STYLE,
    margin=MARGIN
)
fig2.show()

**图2 解读：损失收敛曲线对比（iRprop vs GD）**

本图基于狭长山谷测试函数，展示了 iRprop 与固定学习率 GD 的损失收敛过程（纵轴为对数坐标）：

- **iRprop（红色曲线）**：前 $10$ 步内损失从 $10^0$ 急剧下降至 $10^{-3}$ 量级，随后呈锯齿状持续下降。到第 $50$ 步时，损失已降至 $10^{-11}$ 到 $10^{-13}$ 量级，收敛极快。曲线中的锯齿状波动正是 iRprop 算法中"梯度符号反转"触发步长急刹与回退机制的正常表现。
- **GD（蓝色曲线，lr=0.01）**：$50$ 步后损失仍停留在 $10^{-1}$ 量级，**几乎是一条水平线**。由于固定学习率（$0.01$）远小于平坦方向（$\theta_1$）所需的有效步长，GD 在平坦方向上的推进极为缓慢，导致损失函数值几乎没有下降。

**核心结论**：
iRprop 仅用 $50$ 步便将损失降低了约 $12$ 个数量级（从 $10^0$ 到 $10^{-12}$），而固定学习率的 GD 在相同步数下几乎原地踏步，两者差距极为悬殊。这直观证明了 Rprop/iRprop 利用符号驱动独立步长在确定性梯度下极高的收敛效率。

In [ ]:
# 图3：iRprop 步长演化
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=iters_delta, y=rprop_delta[:, 0], mode='lines+markers', name='Δ₁', line=LINE_RED, marker=MARKER_RED))
fig3.add_trace(go.Scatter(x=iters_delta, y=rprop_delta[:, 1], mode='lines+markers', name='Δ₂ ', line=LINE_BLUE, marker=MARKER_BLUE))
fig3.update_layout(title=dict(text='图3：iRprop 独立步长演化', font=dict(size=18)), xaxis_title='迭代次数', yaxis_title='步长 Δ ', xaxis=AXIS_STYLE, yaxis=LOG_YAXIS_DELTA, width=width, height=h_std, hovermode='x', legend=LEGEND_STYLE, margin=MARGIN)
fig3.show()

**图3 解读：iRprop 独立步长演化**

本图展示两个参数方向独立步长 $\Delta_1$ 和 $\Delta_2$ 随迭代的变化（纵轴为对数坐标）：

- **$\Delta_1$（平坦方向，红色）**：初始约 $10^{-1}$，前 $6$ 步因梯度符号一致而短暂放大（峰值约 $0.3$ 左右），随后因不断越过谷底触发刹车机制，呈现锯齿状持续下降。在 $40$ 步左右降至设定的最小步长下限 $10^{-6}$，此后不再变化。
- **$\Delta_2$（陡峭方向，蓝色）**：初始约 $10^{-1}$，但由于地形过于陡峭，它没有经历短暂的放大过程，而是由于梯度的频繁反转，步长被源源不断地按指数级压缩，一路平滑下降，在 $30$ 步左右就触底达到了最小值 $10^{-6}$。

**关键观察**：
前 $30$ 步中，$\Delta_1$ 始终显著大于 $\Delta_2$，完美匹配了"平坦方向需要大步伐，陡峭方向需要小步伐"的需求。但随着迭代推进，**两个方向最终都因为反复跨越最优解（符号反转）而触发急刹，收缩到了下限 $10^{-6}$**。这说明 iRprop 在保证不震荡的同时，后期也会因为步长过小而陷入极其缓慢的收敛期，最终步长比例趋于 $1:1$。

In [ ]:
# 图4：参数时间序列
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=iters_delta, y=rprop_traj[:, 0], mode='lines+markers', name='iRprop θ₁ (平坦方向，大步长)', line=LINE_RED, marker=MARKER_RED))
fig4.add_trace(go.Scatter(x=iters_delta, y=rprop_traj[:, 1], mode='lines+markers', name='iRprop θ₂ (陡峭方向，小步长)', line=LINE_BLUE, marker=MARKER_BLUE))
fig4.add_hline(y=0, line_dash='solid', line_color='gray', opacity=0.5)
fig4.update_layout(title=dict(text='图4：iRprop 参数轨迹', font=dict(size=18)), xaxis_title='迭代次数', yaxis_title='参数值', xaxis=AXIS_STYLE, yaxis=AXIS_STYLE, width=width, height=h_std, hovermode='x', legend=LEGEND_STYLE, margin=MARGIN)
fig4.show()

**图4 解读：iRprop 参数轨迹**

本图展示 iRprop 算法下两个参数的演化过程（纵轴为参数值）：

- **$\theta_1$（平坦方向，红色）**：初始值为 $1.0$。由于平坦方向梯度较小，步长 $\Delta_1$ 在初期被迅速放大，导致参数以极快的速度下降并在第 $6$ 步左右**直接越过了谷底**，跌落至 $-0.2$ 附近。随后触发 iRprop 的回退与刹车机制，参数在 $0$ 轴上下呈现轻微的锯齿状震荡，并在 $15$ 步左右完全收敛并稳定于 $0$。
- **$\theta_2$（陡峭方向，蓝色）**：初始值为 $0.1$。因为该方向极其陡峭，独立步长 $\Delta_2$ 被严格控制得非常小，参数在最初的 $1\sim2$ 步内就迅速且平缓地下降到了 $0$ 附近。之后没有任何大幅震荡，一直稳定贴合 $0$ 轴。

**关键观察**：
iRprop 的独立步长机制导致了两个方向完全不同的收敛姿态：平坦方向由于"步子迈得太大"经历了大幅度的越界与回弹修正（强震荡）；而陡峭方向则因为"步子被压得极短"实现了几乎完美的瞬态收敛（无震荡）。这充分说明了 iRprop 利用符号频率调节步长的动态博弈过程：虽然前期有剧烈波动，但在 $15$ 步后，**两个参数均成功且彻底地稳定在了最优点 $(0,0)$**。

In [ ]:
# 图5：原始 Rprop vs iRprop 对比
rprop_orig_traj, rprop_orig_delta, rprop_orig_loss = rprop_original(theta0)
fig5 = go.Figure()
fig5.add_trace(go.Scatter(x=iterations, y=rprop_loss, mode='lines+markers', name='iRprop', line=LINE_RED, marker=MARKER_RED))
fig5.add_trace(go.Scatter(x=iterations, y=rprop_orig_loss, mode='lines+markers', name='原始 Rprop', line=LINE_BLUE, marker=MARKER_BLUE))
fig5.update_layout(title=dict(text='图5：原始Rprop vs iRprop 损失收敛对比', font=dict(size=18)), xaxis_title='迭代次数', yaxis_title='Loss (对数坐标)', xaxis=AXIS_STYLE, yaxis=LOG_YAXIS_LOSS, width=width, height=h_std, hovermode='x', legend=LEGEND_STYLE, margin=MARGIN)
fig5.show()

**图5 解读：原始Rprop vs iRprop 损失收敛对比**

本图揭示了原始 Rprop 与 iRprop 在损失下降过程中的本质区别：

- **iRprop（红色实线）**：损失整体呈锯齿状稳步下降。在 $30$ 步后，损失降至约 $10^{-11}$，虽然仍有微小的波动，但被严格限制在较窄的区间（约 $10^{-10}$ 到 $10^{-12}$ 之间），表明算法已趋近稳定。
- **原始 Rprop（蓝色虚线）**：在 $30$ 步后，损失曲线出现了**极其剧烈、跨度达 $3$ 个数量级的震荡**，在 $10^{-11}$ 和 $10^{-14}$ 之间来回跳跃，并且某些点的损失数值甚至比 iRprop 更低。

**核心结论**：
原始 Rprop 之所以会显现出"更低"的损失值（如接近 $10^{-14}$），并不是因为它真正收敛到了最优点，而是因为**缺乏回退机制，导致参数在谷底附近以极其粗暴的方式来回跳跃**，恰好跳到了损失更低的离散点上。这种剧烈的震荡意味着参数极不稳定，无法真正"停"在最优解。
相比之下，iRprop 的损失虽不是全局最低，但波动范围极小，参数真正实现了稳定。**优化器追求的不应是单点极低的 Loss 数字，而是整个优化过程的平稳与参数最终位置的真实收敛。**


从图5的损失收敛对比来看，原始 Rprop 在第48步的损失（~$2\times10^{-8}$）似乎比 iRprop（~$1\times10^{-9}$）更低。

**但这是一个致命陷阱！**

| 指标 | iRprop | 原始 Rprop |
|------|-----------------|-----------|
| 第48步损失 | ~$1\times10^{-9}$ | ~$2\times10^{-8}$（**数值更小！**） |
| 第48步 $\theta_1$ | **~$0.0000001$**（稳定接近0） | **~$0.05$**（还在震荡！） |
| 第48步 $\theta_2$ | **~$0.00000005$**（稳定接近0） | **~$0.05$**（还在大幅震荡！） |

**问题本质**：原始 Rprop 的"低损失"是靠参数的**剧烈来回震荡"平均"出来的**，而不是真正走到了谷底。

**原始 Rprop 在陡峭方向的表现**：
1. 第1次跨过谷底 → 刹车（$\Delta\times0.5$）
2. 下一次又跨过 → 再次刹车（$\Delta\times0.5\times0.5$）
3. 步长被连续砍到极小 → 参数"困"在谷底附近
4. 但又因为回退机制缺失，参数在两个方向之间**来回跳跃**
5. 最终：参数在谷底两侧震荡，**从未真正静止**

**iRprop 的表现**：
1. 第1次跨过谷底 → 刹车 + 回退到上一步
2. 参数回到谷底附近
3. 梯度置零 → 下一次不会触发刹车
4. 步长**稳定不变**，参数在谷底附近微调
5. 最终：参数**静静停在谷底**

> **核心结论**：原始 Rprop 的"低损失"是参数震荡的副产品，不是真正收敛的证据。优化器要优化的**不是单步损失值**，而是**参数的最终稳定位置**。

### 5.3 关键观察与结论

根据图3（iRprop 独立步长演化）的实测轨迹，两个参数方向表现出截然不同的步长调节过程：

| 观察维度 | $\Delta_1$（相对平坦方向，红线） | $\Delta_2$（相对陡峭方向，蓝线） |
|----------|----------------|----------------|
| **初始步长** | $0.1$ | $0.1$ |
| **步长峰值** | 约 $0.5$（第 $5$ 步附近） | $0.1$（无增长） |
| **演化趋势** | 先放大，后因跨过目标点呈锯齿状连续衰减 | 因直接跨过谷底，呈平滑对数线性衰减 |
| **步长终值（第 $50$ 步）** | 约 $10^{-6}$ | 约 $10^{-6}$ |
| **稳定期** | 约第 $42$ 步达到下界 | 约第 $32$ 步达到下界 |

**核心结论**：

> **相对关系的前提**：虽然 $\theta_1$（$\Delta_1$）和 $\theta_2$（$\Delta_2$）在原始函数上存在相对的地形差异（一个梯度系数小，一个梯度系数大），但对于 Rprop 设定的初始步长（$\Delta=0.1$）来说，这两个方向在局部尺度上**都相对陡峭**，因为步长总是轻而易举地跨过相对狭小的最优解目标区域，从而导致符号反转。
>
> **$\Delta_1$（相对平坦方向，红线）**：因初始梯度相对较小，触发“踩油门”，步长放大至 $0.5$。但放大后的步长依然轻易跨过了原点，触发“急刹+回退”，随着逼近最优解，最终呈现锯齿状被砍半至下限 $10^{-6}$。
>
> **$\Delta_2$（相对陡峭方向，蓝线）**：因初始梯度相对极大，起步即跨过谷底，完全没有“踩油门”机会，一路“急刹”，平滑收缩至下限 $10^{-6}$。
>
> **最终状态**：在收敛的临界点，由于两个方向都无限逼近原点，残存的步长都会跨越目标点触发符号反转，最终两个方向的步长都被压缩至最小值下限（$\Delta_{\min} = 10^{-6}$）并保持水平。

**核心洞察**：平坦和陡峭是相对的，但 Rprop 的步长收缩机制是绝对的。无论梯度大小如何，只要步长跨越了目标点（触发符号反转），就会无条件执行收缩。因此，在最终的收敛点上，所有方向的步长都会一致收缩到下限，这正是 Rprop 依靠“符号一致性与反转频率”自适应调节的体现。

## 6. 总结

| 维度 | 结论 |
|------|------|
| **全称** | Resilient Backpropagation（弹性反向传播），Riedmiller & Braun, 1993 |
| **解决的问题** | 统一学习率无法适配不同参数尺度 |
| **核心设计** | 每个参数独立维护步长 $\Delta_i$，仅用梯度符号 $\text{sign}(g_i)$ 决定方向 |
| **与 GD 的关键区别** | Rprop: $\Delta\theta = -\text{sign}(g)\times\Delta$（只用符号）；GD: $\Delta\theta = -\alpha\times g$（用完整梯度） |
| **步长更新公式** | 符号一致：$\Delta = \Delta \times \eta^+$（$\eta^+>1$，踩油门）；符号反转：$\Delta = \Delta \times \eta^-$（$\eta^-<1$，踩刹车） |
| **参数更新公式** | Rprop：$\Delta\theta = -\text{sign}(g) \times \Delta$；GD：$\Delta\theta = -\alpha \times g$ |
| **原始 Rprop vs iRprop** | 原始：只刹车（$\Delta\times\eta^-$），不回退不置零 → 参数震荡，虚假收敛；iRprop：三步制动（$\Delta\times\eta^-$ + 回退 + 置零）→ 真正收敛到谷底 |
| **理论收敛性** | 能保证收敛到梯度为零的临界点（可能是局部极小点或鞍点）；但对跨越浅谷和逃离鞍点**没有任何理论保证**，完全依赖启发式策略 |
| **典型参数** | $\eta^+=1.2$，$\eta^-=0.5$ |
| **步长调节机制** | **指数试探 + 急刹**：符号一致 → $\times\eta^+$；符号反转 → $\times\eta^-$ + 回退 + 梯度清零 |
| **优势** | 在确定性梯度下快速收敛，不受梯度幅值大小干扰 |
| **致命弱点** | 对梯度噪声极度敏感，噪声下步长迅速收缩至 $0$；且无法在理论上保证跨越鞍点或快速穿越浅谷 |
| **适用场景** | 全批量梯度下降 + 确定性目标函数 |
| **深度学习** | 几乎不用，被 Adagrad / RMSprop / Adam 取代 |

---

> **一句话总结**：Rprop 是自适应学习率的先驱，它用"梯度符号一致性"驱动独立步长——在确定性梯度下表现优异，但因对噪声零容忍而未能进入深度学习的主流工具箱。其改进版 iRprop 通过三步制动（急刹 + 回退 + 置零）解决了原始版本的震荡问题，是实际应用中推荐的版本。但无论在理论还是实践中，它都无法保证对鞍点和浅谷地形的有效穿越。

## 附录 A：Rprop 与 Adagrad 的核心区别总结

本附录基于正文第 2 节"Rprop 在演进链条中的位置"展开，系统对比 Rprop 与 Adagrad 的设计差异。

| 比较维度 | **Rprop** | **Adagrad** |
| :--- | :--- | :--- |
| **梯度依赖** | **仅依赖梯度的符号**（正负号），完全忽略梯度的模长。 | **依赖梯度的实际数值**（模长），梯度越大，该维度的惩罚越重。 |
| **学习率衰减** | **不衰减**。学习率在上下界之间波动（增大或减小），但永远不会趋于 $0$。 | **持续衰减**。由于分母是历史梯度的平方和累加，随着训练进行，学习率会单调递减，最终趋于 $0$。 |
| **适用场景** | **全批次（Full-Batch）训练**。主要用于标准的批量梯度下降，梯度稳定时效果极佳。 | **稀疏数据（Sparse Data）**。非常适合 NLP 和推荐系统，因为频繁出现的特征学习率小，稀疏特征（罕见词）学习率大，能抓住关键信息。 |
| **Mini-Batch 兼容性** | **极差**。由于小批量梯度噪声大，符号变化剧烈，Rprop 在 Mini-Batch 下几乎失效（会被震荡干扰）。 | **良好**。天然为小批量设计，能平滑处理梯度噪声。 |

### 为什么 Rprop 不能在 Mini-Batch 下工作？

Rprop 的设计前提是**全局梯度稳定**（全量数据计算出的梯度）。如果换成小批量，由于样本采样噪声，相邻两次梯度的符号**频繁震荡**，导致 Rprop 的"符号判断"机制不断触发"学习率减小"逻辑，最终使得学习率全部缩到最小值，网络无法收敛。

**正是因为 Rprop 在小批量上的失败**，研究者才设计出了 **RMSprop**（Hinton 的课程笔记）和 **Adam**。你可以把 **Adam 看作是 Rprop 的符号思想 + Adagrad 的数值缩放思想的结合体**（即同时利用一阶矩和二阶矩）。

### 优缺点总结

- **Adagrad 的优点**：无需手动调整学习率，适合稀疏梯度。
- **Adagrad 的缺点**：学习率**过早且过度衰减**（因为 $G_t$ 不断累积），训练到后期几乎停止学习。这也正是后来 **Adadelta** 和 **RMSprop** 提出来的原因（它们用滑动平均代替累加）。

- **Rprop 的优点**：在批处理任务中收敛极快，且不受梯度爆炸（Gradient Explosion）影响（因为忽略大小）。
- **Rprop 的缺点**：**无法应用于现代深度学习的主流训练方式（Mini-batch SGD）**，且对超参数（学习率增减上下限）较为敏感。

### 形象类比

- **Adagrad** 像一位**记账的管家**：它详细记录每件物品（参数）过去的开销总额（历史梯度平方和），开销大的地方（大梯度）以后就少批预算（小学习率），开销小的地方多批预算。但管家记账总额越来越大，最后所有预算都趋近于零（停止学习）。
- **Rprop** 像一位**只看风向的船长**：他完全不关心风浪大小（梯度数值），只关心风是从左吹还是从右吹（梯度符号）。如果连续两天风向右，他就加速前行；如果风向突然反向，他就立刻减速。但这种操作只适合风平浪静（全批次）的大海，遇上惊涛骇浪（小批量）他就晕头转向了。

**总结**：如今深度学习中，**你几乎不会单独使用 Rprop**（除非在做传统的全量训练），而 Adagrad 虽然存在衰减问题，但它的思想（自适应学习率）被 Adam 等主流优化器继承并改良，依然活跃在第一线。如果你现在训练神经网络，通常直接选用 Adam 或 AdamW，它们已经巧妙规避了以上两者的极端缺陷。

## 附录 B：PyTorch 中的实现状态

是的，**二者在 PyTorch 中都有实现**，你可以在 `torch.optim` 模块下直接调用它们。

### ✅ Rprop (弹性反向传播)

PyTorch 提供了完整的 `Rprop` 优化器类以及它的函数式接口。

*   **调用方式**：`torch.optim.Rprop(params, lr=0.01, etas=(0.5, 1.2), step_sizes=(1e-6, 50))`。其中 `etas` 和 `step_sizes` 分别控制学习率的增减因子与步长范围。
*   **重要限制**：**Rprop 不支持稀疏梯度（Sparse Gradients）**。如果你的模型涉及稀疏更新（比如某些嵌入层），使用它时会直接报错。

> 不过，如我们上次聊到的，Rprop 的设计初衷是针对全批次（Full-Batch）梯度下降，在如今主流的 Mini-Batch 训练中表现不稳定，因此**实际项目中使用较少**。

### ✅ Adagrad (自适应梯度算法)

Adagrad 同样是 PyTorch 的标准成员，提供了类 `Adagrad` 和对应的函数式接口。

*   **调用方式**：`torch.optim.Adagrad(params, lr=0.01, lr_decay=0, weight_decay=0, initial_accumulator_value=0, eps=1e-10)`。参数如 `lr_decay` 控制学习率衰减，`weight_decay` 用于 L2 正则化。
*   **演进与优化**：为了提升性能，PyTorch 的 Adagrad 实现也在不断优化。例如，新版本增加了 `fused` 参数（目前仅支持 CPU），可以将多个参数的更新操作合并，以提升执行速度。

> 尽管 Adagrad 自身存在学习率后期过快衰减的问题，但作为自适应学习率算法的基石，它仍然是 `torch.optim` 工具箱中一个可靠的选择，尤其在处理稀疏特征数据时。

### 总结

在 PyTorch 中，两者都可以通过 `torch.optim` 直接使用。但如果你问在实际开发中谁更"主流"，**那无疑是 Adagrad**。它虽然不如 Adam 那么通用，但在特定场景（如推荐系统、NLP 中的稀疏特征）下仍有价值；而 Rprop 则更像一个为了算法完整性而保留的选项，在日常训练中比较少见。

## 附录 C：为什么 Rprop 不能用于 Mini-Batch 随机梯度

本附录是对正文第 2 节"核心定位"中提及的致命局限——**不能用于 Mini-Batch 随机梯度**——的深度剖析。

### C.1 根本原因：Rprop 依赖"梯度符号的一致性"来做出决策

Rprop 的步长更新规则完全基于一个判断：

$$
\text{若 } g_t \times g_{t-1} > 0 \Rightarrow \Delta \text{ 放大} \quad \text{若 } g_t \times g_{t-1} < 0 \Rightarrow \Delta \text{ 缩小}
$$

这个判断的**前提假设**是：**梯度的符号变化真实反映了地形特征**——符号持续一致意味着"还没到达谷底"，符号反转意味着"跨过了谷底"。

**但在 Mini-Batch 中，这个假设被彻底破坏了。**

### C.2 核心问题的直观理解

#### C.2.1 全批次（Full-Batch）中的情况

全批次梯度的计算使用**所有训练样本**：

$$
g_{\text{full}} = \frac{1}{N}\sum_{i=1}^{N} \nabla \mathcal{L}_i(\theta)
$$

这个梯度是**确定性的**——在给定参数点，它的值固定不变。因此：

- 如果参数确实在"下坡"的路上，梯度符号会**持续一致**（比如一直为负）；
- 只有真正跨过谷底时，梯度符号才会**准确反转**。

Rprop 的"符号判断"是**可信的**。

#### C.2.2 Mini-Batch 中的情况

Mini-Batch 梯度的计算只使用**一小部分样本**：

$$
g_{\text{batch}} = \frac{1}{B}\sum_{i \in \mathcal{B}} \nabla \mathcal{L}_i(\theta)
$$

关键区别在于：**这个梯度的值取决于你抽到了哪些样本**。每次迭代抽取的 Batch 不同，梯度就会**随机波动**。

**举例说明**（一个简单的线性回归场景）：

| 迭代 | 抽样 Batch | 计算出的梯度 | 符号 |
|------|-----------|-------------|------|
| 第 $t$ 次 | 样本 1, 5, 8 | $g_t = -0.3$ | 负 |
| 第 $t+1$ 次 | 样本 2, 6, 9 | $g_{t+1} = +0.1$ | 正 |

**这里梯度的符号反转让 Rprop 以为"跨过了谷底"，但实际上呢？**

- 参数位置：$\theta = 1.0$，最优解 $\theta^* = 0.5$（还没到）
- 按照全批次计算，真实的梯度应该一直为负（因为 $\theta > \theta^*$）
- 但在第 $t+1$ 次，因为抽到了"幸运"的样本组合，计算出的梯度符号变成正的

**这就是问题的根源：Mini-Batch 梯度的符号变化，不能区分"真实地形的转折"和"抽样噪声造成的假反转"。**

### C.3 Rprop 在 Mini-Batch 下的灾难性行为

#### C.3.1 频繁的"假反转"导致步长被连续砍半

Rprop 的核心决策逻辑是：

| 符号变化 | Rprop 的反应 | 后果 |
|---------|-------------|------|
| 一致（$>0$） | 步长放大 $\times 1.2$ | 正常加速 |
| 反转（$<0$） | **步长缩小 $\times 0.5$** | 刹车 |
| **随机频繁反转** | **步长被反复砍半** | **步长迅速崩落到最小值！** |

用数学来描述：

$$
\Delta_{t+1} = \Delta_t \times (1.2)^{k_+} \times (0.5)^{k_-}
$$

其中 $k_+$ 是符号一致次数，$k_-$ 是符号反转次数。如果符号**随机翻转**（每次迭代约 50% 概率反转），那么：

$$
\mathbb{E}[\Delta_{t+1}] \approx \Delta_t \times 1.2^{0.5} \times 0.5^{0.5} \approx \Delta_t \times 0.775
$$

这意味着：**步长以平均每步 22.5% 的速度指数衰减**。

| 迭代次数 | 步长（初始 0.1） |
|---------|----------------|
| 0 | 0.1 |
| 10 | $0.1 \times 0.775^{10} \approx 0.008$ |
| 50 | $0.1 \times 0.775^{50} \approx 1.8 \times 10^{-6}$ |
| 100 | **$\approx 0$（触碰到下限 $10^{-6}$）** |

**结果：不出 100 步，所有参数的步长都会收缩到最小值 $\Delta_{\min} = 10^{-6}$，算法彻底"死掉"！**

#### C.3.2 更致命的问题：Rprop 无法区分"地形转折"与"噪声抖动"

我们用一张表来对比不同场景下 Rprop 的行为：

| 场景 | 真实的梯度符号变化 | Rprop 的解读 | 正确行为应该是什么？ | Rprop 实际行为 |
|------|-------------------|-------------|-------------------|---------------|
| 全批次：参数已跨过谷底 | $g_{t-1} < 0$, $g_t > 0$ | "跨过了谷底" → 刹车+回退 | 缩小步长，回退到谷底附近 | ✅ 正确 |
| Mini-Batch：参数未到谷底，但抽样噪声造成反转 | $g_{t-1} < 0$, $g_t > 0$（假） | **误判为"跨过了谷底"** → 刹车+回退 | 继续前进（没到目标） | ❌ 错误！ |
| Mini-Batch：参数已跨过谷底，但抽样噪声造成"一致" | $g_{t-1} < 0$, $g_t < 0$（假） | **误判为"还没到谷底"** → 踩油门 | 缩小步长，不要继续前进 | ❌ 错误！ |

**结论**：在 Mini-Batch 环境下，Rprop 的"符号判断"从一个**可靠的地形探测器**，退化为一个**随机的步长控制器**。它既可能在需要加速时误刹车，也可能在需要刹车时误加速。

### C.4 数学分析：为什么噪声会"毒死"符号判断

#### C.4.1 信号噪声比（SNR）的角度

设真实梯度为 $\bar{g}$，Mini-Batch 梯度为：

$$
g = \bar{g} + \epsilon
$$

其中 $\epsilon$ 是抽样噪声。当 Batch 很小（如 32~128）时，$\epsilon$ 的方差可能**远大于** $\bar{g}$ 的幅值，特别是在训练后期。

定义**信号噪声比**：

$$
\text{SNR} = \frac{|\bar{g}|}{\sqrt{\mathbb{E}[\epsilon^2]}}
$$

当 SNR 较小时：

- $g$ 的符号几乎完全由噪声 $\epsilon$ 决定，**与真实梯度无关**；
- Rprop 的符号判断完全失去意义；
- 步长更新变成一个**随机游走**——但并不是对称的，因为"刹车"的幂（×0.5）远大于"加速"的幂（×1.2）。

#### C.4.2 不等式分析

设符号反转的概率为 $p$（在纯噪声下约 $0.5$），符号一致的概率为 $1-p$。

经过 $T$ 次迭代后，步长的**期望值**为：

$$
\mathbb{E}[\Delta_T] = \Delta_0 \times (\eta^+)^{(1-p)T} \times (\eta^-)^{pT}
$$

代入典型值 $\eta^+ = 1.2, \eta^- = 0.5, p = 0.5$：

$$
\mathbb{E}[\Delta_T] = \Delta_0 \times (1.2 \times 0.5)^{0.5T} = \Delta_0 \times 0.6^{0.5T}
$$

**指数收敛到 0！** 只要 $\eta^+ \times \eta^- < 1$（在典型超参数下，$1.2 \times 0.5 = 0.6 < 1$），步长就会以**指数速度**崩落。

### C.5 对比：为什么其他自适应算法能容忍噪声？

| 算法 | 如何处理梯度 | 为什么能容忍 Mini-Batch 噪声？ |
|------|-------------|-------------------------------|
| **Rprop** | 只用梯度的**符号**（±1） | 符号是**离散的**，对噪声**极度敏感**——一点点噪声就能把符号翻转 |
| **Adagrad** | 用梯度的**平方累加** | 梯度平方是**连续值**，噪声会被**平滑**到累加和中，不会造成剧烈震荡 |
| **RMSprop** | 用梯度平方的**滑动平均** | 滑动平均天然**平滑噪声**，且步长是连续的，不会像 Rprop 那样发生跳变 |
| **Adam** | 用一阶矩 + 二阶矩的**滑动平均** | 同时利用梯度的大小和方差，抗噪性最强 |

### C.6 一句话总结

> **Rprop 的"只使用符号"策略在确定性梯度下是优雅的，但在随机梯度下是灾难性的。符号作为"二值化的梯度信息"，对噪声的敏感度是无与伦比的——哪怕是最微小的抽样波动，也会让符号在正负之间乱跳，导致步长被高频触发的"刹车"机制以指数级速度砍缩至零，算法在噪声中自毁。**

### C.7 实验验证参考

如果你希望亲自验证这一点，可以做一个简单的实验：

1. 定义一个简单的二次函数（如 $f(\theta) = \theta^2$），最优解在 $\theta=0$；
2. 对全批次梯度运行 Rprop → **正常收敛**；
3. 对带噪声的梯度（每次迭代在真实梯度上添加 $\mathcal{N}(0, 0.1)$ 的随机噪声）运行 Rprop → **步长迅速崩落到 $\Delta_{\min}$，参数永远停在初始点附近，无法收敛**。

这正是 Rprop 之所以**在现代深度学习（几乎全部使用 Mini-Batch 训练）中被完全淘汰**的根本原因。它的"符号驱动"哲学虽然在确定性优化中创造过辉煌，但在充满噪声的随机优化时代，只能让位于更稳健的算法（如 Adam 及其改进版本）。